# Módulo CAR · Retos de entrevista — Práctica guiada

Cinco retos que reproducen lo que te encontrarás en procesos reales (ver `README.md` y el plan v3, sección 2.1):

| Reto | Tipo de prueba | Tiempo orientativo |
|---|---|---|
| 1 | *Live coding* con pandas | 10 min |
| 2 | *Live coding* con SQL (SQLite) | 10 min |
| 3 | Algoritmos sencillos con tests `assert` | 10 min |
| 4 | Diseño de un sistema de ML, guiado | 20 min |
| 5 | Análisis de una oferta: requisitos imprescindibles frente a deseables | 15 min |

**Cómo usarlo (importante).** Cada reto tiene una celda para tu solución y otra que la comprueba. Las comprobaciones **nunca lanzan error**: te dicen `OK`, `FALLA` o `PENDIENTE`, igual que un entrevistador te diría «casi».

**Dos modos de práctica**, porque hay procesos de los dos tipos (Provectus permite asistente de IA en el *live coding*; ICF no lo contempla):
- **Modo sin IA:** cronómetro, sin buscar nada salvo la documentación oficial. Habla en voz alta como si te evaluaran.
- **Modo con IA:** puedes usar un asistente, pero apunta en el «diario» del final qué le pediste y qué corregiste. Te evalúan por tu criterio, no por el texto que pega la herramienta.

Todo se ejecuta en CPU en segundos y sin clave de API.

In [ ]:
import re
import sqlite3
import time

import numpy as np
import pandas as pd


class Cronometro:
    """Cronómetro para practicar con tiempo. No interrumpe nada: solo te dice cómo vas."""
    def iniciar(self, minutos: int) -> None:
        self.limite, self.t0 = minutos * 60, time.time()
        print(f"Cronómetro en marcha: {minutos} min. ¡Ahora!")

    def parar(self) -> None:
        usado = time.time() - self.t0
        estado = "dentro de tiempo" if usado <= self.limite else "te has pasado: en la entrevista, di qué harías con más tiempo"
        print(f"Tiempo usado: {usado / 60:.1f} min ({estado}).")

reloj = Cronometro()
RESULTADOS = {}


def comprobar(nombre, prueba) -> None:
    """Ejecuta una prueba sin lanzar nunca excepciones: OK / FALLA / PENDIENTE."""
    try:
        prueba()
    except NotImplementedError:
        RESULTADOS[nombre] = "PENDIENTE"
    except AssertionError as e:
        RESULTADOS[nombre] = "FALLA"
        print(f"  FALLA en «{nombre}»: {e}")
    except Exception as e:                                   # un error en tu código también es información
        RESULTADOS[nombre] = "FALLA"
        print(f"  ERROR en «{nombre}»: {type(e).__name__}: {e}")
    else:
        RESULTADOS[nombre] = "OK"
    print(f"{RESULTADOS[nombre]:9s} {nombre}")

In [ ]:
# Datos sintéticos de TiendaSur (comercio de café y menaje), con semilla para que todos veáis lo mismo
rng = np.random.default_rng(7)
ciudades = ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao"]
productos = pd.DataFrame({
    "id_producto": range(1, 13),
    "nombre": ["Café Arábica 1kg", "Café Robusta 1kg", "Descafeinado 500g", "Molinillo manual", "Cafetera italiana",
               "Prensa francesa", "Taza cerámica", "Set 4 tazas", "Tetera", "Filtros V60", "Báscula barista", "Jarra espumadora"],
    "categoria": ["cafe"] * 3 + ["cafetera"] * 3 + ["menaje"] * 3 + ["accesorio"] * 3,
    "precio": [14.5, 11.0, 7.5, 32.0, 24.0, 19.0, 6.0, 21.0, 17.0, 5.5, 28.0, 15.0],
})
clientes = pd.DataFrame({
    "id_cliente": range(1, 201),
    "ciudad": rng.choice(ciudades, 200, p=[.3, .25, .2, .15, .1]),
    "alta": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 600, 200), unit="D"),
})
n = 2500
pedidos = pd.DataFrame({
    "id_pedido": range(1, n + 1),
    "id_cliente": rng.integers(1, 201, n),
    "id_producto": rng.integers(1, 13, n),
    "fecha": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 295, n), unit="D"),
    "unidades": rng.integers(1, 4, n),
})
pedidos["importe"] = (pedidos["unidades"] * pedidos["id_producto"].map(productos.set_index("id_producto")["precio"])).round(2)

con = sqlite3.connect(":memory:")
for nombre, tabla in {"pedidos": pedidos.assign(fecha=pedidos["fecha"].dt.strftime("%Y-%m-%d")),
                      "clientes": clientes.assign(alta=clientes["alta"].dt.strftime("%Y-%m-%d")),
                      "productos": productos}.items():
    tabla.to_sql(nombre, con, index=False)

print(pedidos.shape, clientes.shape, productos.shape)
pedidos.head()

## Reto 1 · pandas (10 min)

**Enunciado.** Marketing quiere los clientes que más han gastado recientemente.

Escribe `top_clientes(pedidos, n=3, hoy="2026-10-23", dias=90)` que devuelva un `DataFrame` con las columnas `id_cliente` y `gasto` (suma de `importe`) de los `n` clientes con mayor gasto en la ventana `(hoy - dias, hoy]` (excluye el límite inferior, incluye `hoy`).

Reglas: orden por `gasto` descendente y, en caso de empate, por `id_cliente` ascendente; índice reiniciado; si no hay pedidos en la ventana, devuelve un DataFrame vacío **con las dos columnas**.

*Qué mide el entrevistador:* `groupby`, filtrado por fechas, orden estable y casos límite (¿pensaste en el vacío y en los empates?).

In [ ]:
reloj.iniciar(10)

def top_clientes(pedidos, n=3, hoy="2026-10-23", dias=90):
    hoy = pd.Timestamp(hoy)
    desde = hoy - pd.Timedelta(days=dias)
    p = pedidos.assign(fecha=pd.to_datetime(pedidos["fecha"]))
    p = p[(p["fecha"] > desde) & (p["fecha"] <= hoy)]                       # ventana abierta por abajo
    gasto = p.groupby("id_cliente", as_index=False)["importe"].sum().rename(columns={"importe": "gasto"})
    return (gasto.sort_values(["gasto", "id_cliente"], ascending=[False, True])   # desempate determinista
                 .head(n).reset_index(drop=True))

In [ ]:
def prueba_r1():
    p = pd.DataFrame({
        "id_cliente": [1, 2, 2, 3, 4, 5],
        "fecha": ["2026-10-23", "2026-10-01", "2026-09-01", "2026-07-25", "2026-07-26", "2026-01-01"],
        "importe": [50.0, 30.0, 20.0, 99.0, 50.0, 500.0],
    })
    # 2026-07-25 es exactamente hoy-90: queda FUERA; 2026-07-26 queda DENTRO; el cliente 5 queda fuera.
    r = top_clientes(p, n=3)
    assert list(r.columns) == ["id_cliente", "gasto"], f"columnas {list(r.columns)}"
    assert r["gasto"].tolist() == [50.0, 50.0, 50.0], f"gastos {r['gasto'].tolist()}"
    # empate a 50: 1, 2 (30+20) y 4 -> desempate por id ascendente
    assert r["id_cliente"].tolist() == [1, 2, 4], f"desempate: {r['id_cliente'].tolist()}"
    assert list(r.index) == [0, 1, 2], "índice sin reiniciar"
    assert len(top_clientes(p, n=10)) == 3, "n mayor que los clientes disponibles"
    vacio = top_clientes(p, n=3, hoy="2025-06-01")
    assert len(vacio) == 0 and list(vacio.columns) == ["id_cliente", "gasto"], "caso vacío"
    real = top_clientes(pedidos, n=5)
    assert len(real) == 5 and real["gasto"].is_monotonic_decreasing


comprobar("Reto 1 · top_clientes", prueba_r1)

In [ ]:
reloj.parar()

## Reto 2 · SQL (10 min)

**Enunciado.** Para cada categoría de producto, muestra los **2 productos con más ingresos** (suma de `importe`).

Devuelve las columnas `categoria`, `producto` (el `nombre`), `ingresos` (redondeado a 2 decimales) y `posicion` (1 o 2), ordenadas por `categoria` y `posicion`. Desempate por `producto` en orden alfabético.

La conexión `con` (SQLite) ya tiene las tablas `pedidos`, `clientes` y `productos`. *Pista de método:* agrega primero, numera después (`ROW_NUMBER() OVER (PARTITION BY ...)`) y filtra al final. Se hace con una CTE.

*Qué mide el entrevistador:* `JOIN`, `GROUP BY`, funciones de ventana y orden de las operaciones.

In [ ]:
reloj.iniciar(10)

SQL_R2 = """
WITH ingresos AS (
    SELECT p.categoria, p.nombre AS producto, ROUND(SUM(o.importe), 2) AS ingresos
    FROM pedidos AS o
    JOIN productos AS p ON p.id_producto = o.id_producto
    GROUP BY p.categoria, p.nombre
),
numerado AS (
    SELECT categoria, producto, ingresos,
           ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY ingresos DESC, producto) AS posicion
    FROM ingresos
)
SELECT categoria, producto, ingresos, posicion
FROM numerado
WHERE posicion <= 2
ORDER BY categoria, posicion
"""
pd.read_sql_query(SQL_R2, con)

In [ ]:
def prueba_r2():
    if not SQL_R2.strip() or SQL_R2.strip().startswith("--") and len(SQL_R2.strip().splitlines()) == 1:
        raise NotImplementedError
    obtenido = pd.read_sql_query(SQL_R2, con)
    assert list(obtenido.columns) == ["categoria", "producto", "ingresos", "posicion"], f"columnas {list(obtenido.columns)}"
    # Verdad calculada con pandas, por un camino distinto
    ing = (pedidos.merge(productos, on="id_producto").groupby(["categoria", "nombre"], as_index=False)["importe"].sum()
           .rename(columns={"nombre": "producto", "importe": "ingresos"}))
    ing["ingresos"] = ing["ingresos"].round(2)
    ing = ing.sort_values(["categoria", "ingresos", "producto"], ascending=[True, False, True])
    ing["posicion"] = ing.groupby("categoria").cumcount() + 1
    esperado = ing[ing["posicion"] <= 2].sort_values(["categoria", "posicion"]).reset_index(drop=True)
    assert len(obtenido) == len(esperado), f"{len(obtenido)} filas, esperaba {len(esperado)}"
    pd.testing.assert_frame_equal(obtenido.reset_index(drop=True), esperado[obtenido.columns], check_dtype=False, atol=0.01)


comprobar("Reto 2 · SQL top-2 por categoría", prueba_r2)

In [ ]:
reloj.parar()

## Reto 3 · Algoritmos sencillos (10 min)

Dos funciones típicas de pantalla técnica. Hay una condición implícita que casi siempre se evalúa: **el coste**. Un test con 300 000 elementos descubre soluciones de coste cuadrático.

**3a.** `dos_suma(nums, objetivo)`: devuelve `(i, j)` con `i < j` tal que `nums[i] + nums[j] == objetivo`, eligiendo la pareja de **menor `j`**; si no existe, `None`. Debe ser O(n).

**3b.** `mejor_ventana(ventas, k)`: ventas diarias de TiendaSur; devuelve `(suma, inicio)` de los `k` días consecutivos con mayor suma (si hay empate, el primero). Lanza `ValueError` si `k` es mayor que `len(ventas)` o `k < 1`. Debe ser O(n).

*Qué mide el entrevistador:* que expliques la idea (tabla hash / ventana deslizante) **antes** de escribir, los casos límite y el coste.

In [ ]:
reloj.iniciar(10)

def dos_suma(nums, objetivo):
    """Idea: guardar lo ya visto en un diccionario valor -> índice; para cada j buscar el complemento. O(n) tiempo, O(n) memoria."""
    vistos = {}
    for j, x in enumerate(nums):
        i = vistos.get(objetivo - x)
        if i is not None:
            return (i, j)
        vistos.setdefault(x, j)                  # setdefault: con duplicados conservamos el primer índice
    return None


def mejor_ventana(ventas, k):
    """Idea: ventana deslizante; al avanzar se suma el que entra y se resta el que sale. O(n)."""
    if k < 1 or k > len(ventas):
        raise ValueError("k debe estar entre 1 y len(ventas)")
    suma = sum(ventas[:k])
    mejor, inicio = suma, 0
    for i in range(k, len(ventas)):
        suma += ventas[i] - ventas[i - k]
        if suma > mejor:                         # '>' estricto: en un empate se queda el primero
            mejor, inicio = suma, i - k + 1
    return (mejor, inicio)

In [ ]:
def prueba_r3a():
    assert dos_suma([2, 7, 11, 15], 9) == (0, 1)
    assert dos_suma([3, 2, 4], 6) == (1, 2)
    assert dos_suma([3, 3], 6) == (0, 1), "duplicados"
    assert dos_suma([1, 2, 3], 100) is None
    assert dos_suma([], 1) is None
    assert dos_suma([5], 10) is None, "no se puede usar el mismo elemento dos veces"
    grande = list(range(0, 600_000, 2))                      # solo pares: el objetivo impar es imposible
    t0 = time.time()
    assert dos_suma(grande, 1) is None
    assert time.time() - t0 < 2, "demasiado lento: ¿coste cuadrático?"


def prueba_r3b():
    assert mejor_ventana([1, 3, 2, 6, 1, 1], 2) == (8, 2)
    assert mejor_ventana([5, 5, 5], 2) == (10, 0), "empate: el primero"
    assert mejor_ventana([4], 1) == (4, 0)
    assert mejor_ventana([-3, -1, -2], 2) == (-3, 1), "ventas negativas (devoluciones)"
    for k in (0, 4):
        try:
            mejor_ventana([1, 2, 3], k)
        except ValueError:
            continue
        raise AssertionError(f"k={k} debería lanzar ValueError")
    datos = list(np.random.default_rng(0).integers(0, 100, 300_000))
    t0 = time.time()
    mejor_ventana(datos, 1000)
    assert time.time() - t0 < 2, "demasiado lento: ¿recalculas la suma en cada ventana?"


comprobar("Reto 3a · dos_suma", prueba_r3a)
comprobar("Reto 3b · mejor_ventana", prueba_r3b)

In [ ]:
reloj.parar()

## Reto 4 · Diseño de un sistema de ML (20 min, guiado)

**Enunciado de entrevista.** *«TiendaSur pierde clientes. Diseña un sistema que detecte clientes en riesgo de abandono y permita a marketing actuar.»*

En la entrevista tendrás un folio y 30–45 minutos. Aquí lo guiamos en 9 apartados: rellena el diccionario `diseno` con **frases completas** (no etiquetas sueltas). Estructura recomendada, en este orden: negocio → datos → baseline → validación → despliegue → monitorización → riesgos.

| Apartado | Pregunta que tienes que contestar |
|---|---|
| `objetivo_negocio` | ¿Qué decisión cambia el modelo y quién la toma? |
| `metrica_negocio` | ¿Cómo sabremos que ha merecido la pena (en euros o clientes)? |
| `metrica_tecnica` | ¿Qué métrica de modelo y por qué esa y no la exactitud? |
| `datos` | ¿Qué datos hay, de qué periodo y cuál es la **etiqueta** exacta («abandona» = ...)? |
| `baseline` | ¿Con qué se compara? (regla simple, modelo trivial) |
| `validacion` | ¿Cómo se parte los datos sin fugas? |
| `despliegue` | ¿Lote o tiempo real? ¿Cómo llegan las predicciones a marketing? |
| `monitorizacion` | ¿Qué se vigila y cuándo se reentrena? |
| `riesgos` | ¿Qué puede salir mal (sesgos, privacidad, costes de los falsos positivos)? |

La función `evaluar_diseno` no te pone nota: comprueba que cada apartado está contestado y avisa de temas que un entrevistador espera oír. *Léete antes `apuntes.md`, sección «Diseño de sistemas de ML».*

In [ ]:
PALABRAS_ESPERADAS = {
    "metrica_tecnica": (["recall", "precisión", "precision", "auc", "pr-auc", "f1", "coste"], "justifica una métrica adecuada a clases desbalanceadas"),
    "datos": (["etiqueta", "ventana", "periodo", "días", "dias", "meses"], "define la etiqueta con una ventana temporal"),
    "baseline": (["regla", "logístic", "logistic", "ingenu", "mayoría", "recencia", "baseline", "sin modelo"], "propone un punto de comparación simple"),
    "validacion": (["temporal", "fuga", "leakage", "futuro", "tiempo"], "evita fugas: partición por tiempo, no aleatoria"),
    "despliegue": (["lote", "batch", "diari", "semanal", "api", "tiempo real"], "elige lote o tiempo real con un motivo"),
    "monitorizacion": (["deriva", "drift", "psi", "reentren", "etiquetas", "métrica"], "vigila deriva y define cuándo reentrenar"),
    "riesgos": (["sesgo", "privacidad", "rgpd", "falso", "coste", "molest"], "menciona riesgos de negocio y de personas"),
}
APARTADOS = ["objetivo_negocio", "metrica_negocio", "metrica_tecnica", "datos", "baseline", "validacion", "despliegue", "monitorizacion", "riesgos"]


def evaluar_diseno(diseno: dict) -> list[str]:
    """Devuelve la lista de avisos; vacía = diseño completo según estas comprobaciones básicas."""
    avisos = []
    for clave in APARTADOS:
        texto = str(diseno.get(clave, "")).strip()
        if len(texto) < 40:
            avisos.append(f"{clave}: contesta con una o dos frases completas (ahora tiene {len(texto)} caracteres).")
            continue
        if clave in PALABRAS_ESPERADAS:
            claves, consejo = PALABRAS_ESPERADAS[clave]
            if not any(c in texto.lower() for c in claves):
                avisos.append(f"{clave}: no veo que {consejo}.")
    return avisos

In [ ]:
reloj.iniciar(20)

diseno = {
    "objetivo_negocio": "Marketing decide a qué clientes enviar una oferta de retención cada semana; el modelo prioriza a quién, con un presupuesto limitado de ofertas.",
    "metrica_negocio": "Margen retenido: clientes que no abandonan gracias a la oferta por su margen anual, menos el coste de las ofertas; se mide con un grupo de control sin oferta (prueba A/B).",
    "metrica_tecnica": "PR-AUC y recall en el 10 % de clientes con mayor riesgo (el tamaño de la campaña); la exactitud engaña porque abandona una minoría de clientes.",
    "datos": "Pedidos, visitas y tickets de soporte de los últimos 24 meses. Etiqueta: cliente que no compra en los 90 días siguientes a la fecha de corte, habiendo comprado en los 12 meses anteriores.",
    "baseline": "Una regla de recencia (días desde la última compra) y una regresión logística con pocas variables; el modelo complejo solo se justifica si las mejora de forma clara.",
    "validacion": "Partición temporal: entreno con fechas de corte antiguas y valido con las más recientes; las variables se calculan solo con datos anteriores al corte para evitar fugas del futuro.",
    "despliegue": "Predicción en lote cada semana (los clientes no cambian de un día para otro), publicada en una tabla que consume marketing; el modelo empaquetado en una API para reproducibilidad.",
    "monitorizacion": "PSI de las variables clave y de la puntuación cada semana, métrica real cuando llegan las etiquetas a 90 días y reentrenamiento trimestral o si el PSI supera 0,25.",
    "riesgos": "Falsos positivos que reciben descuentos innecesarios (coste), sesgos por zona geográfica, tratamiento de datos personales conforme al RGPD y efecto de molestar al cliente con demasiados correos.",
}
avisos = evaluar_diseno(diseno)
print("Pendiente de completar:" if avisos else "Diseño completo.")
for a in avisos:
    print(" -", a)
RESULTADOS["Reto 4 · diseño de ML"] = "PENDIENTE" if avisos else "OK"

In [ ]:
reloj.parar()

> Después de escribirlo, **cuéntalo en voz alta en 5 minutos**. Un buen diseño con mala explicación puntúa peor que un diseño sencillo bien explicado. Y pregunta siempre antes de diseñar: ¿cuántos clientes?, ¿qué acción hay disponible?, ¿qué pasa si me equivoco?

## Reto 5 · Analizar una oferta (15 min)

**Objetivo.** Antes de preparar una candidatura hay que saber qué es **imprescindible** y qué **deseable**, y qué te falta. Las ofertas mezclan ambas cosas con formatos distintos (ver `apuntes.md`, «Cómo leer una oferta»). Aquí lo automatizamos con reglas simples sobre dos ofertas **sintéticas** inspiradas en las del plan v3, sección 2.1 (no son ofertas reales).

Escribe `analizar_oferta(texto)` que devuelva un diccionario con:
- `"imprescindibles"` y `"deseables"`: **conjuntos de competencias** (nombres canónicos del diccionario `COMPETENCIAS`);
- `"anios"`: el menor número de años de experiencia que aparece (`int`) o `None`.

Reglas:
1. El texto se divide en secciones por líneas de título (terminan en `:`).
2. Un título con `requisitos`, `requirements`, `must have`, `what you bring` o `imprescindible` abre una sección **imprescindible**; con `se valorará`, `valorable`, `deseable`, `nice to have` o `plus`, una **deseable**; cualquier otro título (funciones, qué ofrecemos...) **se ignora**: las competencias que aparecen ahí no son requisitos.
3. Dentro de una sección, un punto que contenga `(valorable)`, `(plus)` o `(deseable)` cuenta como deseable aunque esté en imprescindibles.
4. Si una competencia sale en ambas listas, es imprescindible.

Después, `brecha(oferta, mis_competencias)` te dirá qué te falta, qué es urgente y qué es mejorable.

In [ ]:
COMPETENCIAS = {
    "python": r"\bpython\b", "sql": r"(?<![\w])sql\b", "git": r"\bgit\b", "docker": r"\bdocker\b",
    "kubernetes": r"\bkubernetes\b|\bk8s\b", "aws": r"\baws\b", "azure": r"\bazure\b", "mlflow": r"\bmlflow\b",
    "fastapi": r"\bfastapi\b", "llm": r"\bllms?\b|modelos de lenguaje|language models?", "rag": r"\brag\b",
    "mcp": r"\bmcp\b", "react": r"\breact\b", "typescript": r"\btypescript\b",
    "agentes de código": r"claude code|\bcursor\b", "ci/cd": r"ci/cd|github actions", "inglés": r"inglés|english",
    "evaluación de llms": r"evaluaci[óo]n (?:automatizada )?de (?:sistemas|llms)|\bevals?\b|automated evaluation|evaluation of llm",
}
TITULOS_IMPRESCINDIBLE = ["requisitos", "requirements", "must have", "what you bring", "imprescindible"]
TITULOS_DESEABLE = ["se valorará", "valorable", "deseable", "nice to have", "plus"]
MARCAS_DESEABLE = ["(valorable)", "(plus)", "(deseable)"]


def competencias_en(texto: str) -> set[str]:
    return {nombre for nombre, patron in COMPETENCIAS.items() if re.search(patron, texto, flags=re.IGNORECASE)}


OFERTA_ES = """AI Engineer junior · Madrid · Híbrido

Sobre el puesto:
Trabajarás con producto para llevar asistentes basados en LLM a producción. Construirás demos en React para clientes.

Requisitos imprescindibles:
- Grado en Ingeniería, Matemáticas o similar; 0-2 años de experiencia.
- Python sólido y SQL.
- Experiencia con APIs de LLM y RAG (los proyectos propios valen).
- Git y trabajo con Docker.
- Inglés B2 o superior.
- Despliegue en AWS (valorable).

Se valorará:
- Azure o Kubernetes.
- Experiencia con MLflow.
- Uso diario de Claude Code o Cursor.
- Conocimiento de MCP.

Ofrecemos:
- Plan de formación y mentoría. Equipo con CI/CD y revisión de código.
"""

OFERTA_EN = """Forward Deployed Engineer (remote, Spain)

What you'll do:
Build integrations for customers using Python and TypeScript.

What you bring:
- 2+ years building software; strong Python and FastAPI.
- Hands-on experience with LLMs and automated evaluation of LLM systems.
- TypeScript and React for customer-facing demos.
- Fluent English.

Nice to have:
- Kubernetes, AWS.
- Experience with MCP servers.
"""

In [ ]:
reloj.iniciar(15)

def analizar_oferta(texto: str) -> dict:
    imprescindibles, deseables, anios = set(), set(), []
    seccion = None                                   # None = sección que se ignora
    for linea in texto.splitlines():
        limpia = linea.strip()
        if not limpia:
            continue
        if limpia.endswith(":") and not limpia.startswith("-"):          # línea de título
            titulo = limpia.lower()
            if any(t in titulo for t in TITULOS_DESEABLE):
                seccion = "deseable"
            elif any(t in titulo for t in TITULOS_IMPRESCINDIBLE):
                seccion = "imprescindible"
            else:
                seccion = None
            continue
        for m in re.finditer(r"(\d+)(?:\s*[-–]\s*\d+)?\+?\s*(?:años|years)", limpia, flags=re.IGNORECASE):
            anios.append(int(m.group(1)))                                 # el menor: es lo que de verdad exigen
        if seccion is None:
            continue
        encontradas = competencias_en(limpia)
        es_deseable = seccion == "deseable" or any(marca in limpia.lower() for marca in MARCAS_DESEABLE)
        (deseables if es_deseable else imprescindibles).update(encontradas)
    return {"imprescindibles": imprescindibles, "deseables": deseables - imprescindibles,
            "anios": min(anios) if anios else None}


def brecha(oferta: dict, mis_competencias: set[str]) -> dict:
    imp = oferta["imprescindibles"]
    return {
        "urgente": imp - mis_competencias,
        "mejorable": oferta["deseables"] - mis_competencias,
        "cobertura": len(imp & mis_competencias) / len(imp) if imp else 1.0,
    }

In [ ]:
def prueba_r5():
    a = analizar_oferta(OFERTA_ES)
    assert a["imprescindibles"] >= {"python", "sql", "llm", "rag", "git", "docker", "inglés"}, f"imprescindibles: {sorted(a['imprescindibles'])}"
    assert a["deseables"] >= {"aws", "azure", "kubernetes", "mlflow", "agentes de código", "mcp"}, f"deseables: {sorted(a['deseables'])}"
    assert "aws" not in a["imprescindibles"], "'(valorable)' dentro de imprescindibles es deseable"
    assert not (a["imprescindibles"] & a["deseables"]), "una competencia no puede estar en ambas listas"
    assert "react" not in a["imprescindibles"] | a["deseables"], "las 'funciones' no son requisitos"
    assert "ci/cd" not in a["imprescindibles"] | a["deseables"], "'Ofrecemos' no es un requisito"
    assert a["anios"] == 0, f"anios={a['anios']}"
    b = analizar_oferta(OFERTA_EN)
    assert b["imprescindibles"] >= {"python", "fastapi", "llm", "evaluación de llms", "typescript", "react", "inglés"}, f"imprescindibles EN: {sorted(b['imprescindibles'])}"
    assert b["deseables"] == {"kubernetes", "aws", "mcp"}, f"deseables EN: {sorted(b['deseables'])}"
    assert b["anios"] == 2


def prueba_r5_brecha():
    mis = {"python", "sql", "git", "llm", "inglés", "mlflow"}
    g = brecha(analizar_oferta(OFERTA_ES), mis)
    assert g["urgente"] == {"rag", "docker"}, f"urgente: {sorted(g['urgente'])}"
    assert "mlflow" not in g["mejorable"] and "mcp" in g["mejorable"], f"mejorable: {sorted(g['mejorable'])}"
    assert 0 < g["cobertura"] < 1 and abs(g["cobertura"] - 5 / 7) < 1e-9, f"cobertura {g['cobertura']}"


comprobar("Reto 5a · analizar_oferta", prueba_r5)
comprobar("Reto 5b · brecha", prueba_r5_brecha)

In [ ]:
MIS_COMPETENCIAS = {"python", "sql", "git", "llm", "inglés", "mlflow"}      # cámbialo por las tuyas, con honestidad
try:
    for nombre, oferta in {"ES": OFERTA_ES, "EN": OFERTA_EN}.items():
        analisis = analizar_oferta(oferta)
        g = brecha(analisis, MIS_COMPETENCIAS)
        print(f"Oferta {nombre}: cobertura de imprescindibles {g['cobertura']:.0%}")
        print("  urgente  :", sorted(g["urgente"]))
        print("  mejorable:", sorted(g["mejorable"]))
except NotImplementedError:
    print("Implementa analizar_oferta y brecha para ver tu informe de brecha.")

In [ ]:
reloj.parar()

## Diario del modo «con IA»

Repite el **reto 1 o el 3b** en modo con IA y completa esta tabla en tu cuaderno (es lo que cuenta una persona que sabe trabajar con asistentes):

| Qué le pedí | Qué devolvió | Qué comprobé (test, caso límite, coste) | Qué cambié y por qué |
|---|---|---|---|
| | | | |

Dos preguntas de control, las mismas que te harán: **¿por qué has elegido esta solución?** y **¿qué falla con una entrada vacía o enorme?**

In [ ]:
print("Resumen de la sesión")
for nombre, estado in RESULTADOS.items():
    print(f"  {estado:9s} {nombre}")
assert RESULTADOS and all(e == "OK" for e in RESULTADOS.values()), RESULTADOS   # las soluciones deben pasar todas

## Cierre

- Si algo salió `FALLA`: lee el mensaje, razona la causa y corrige **sin mirar** `soluciones.ipynb` (la entrevista no tiene solucionario).
- Guarda tus soluciones en un repositorio `retos-entrevista` con un README: es una pieza de portafolio menor pero útil, y te sirve de repaso.
- Siguiente paso: la sesión de simulacro con `plantillas/banco-preguntas-entrevista.md`.